# Preprocessing des événements — préparation à l'indexation

**Objectif :** transformer `data/raw/events.csv` en un corpus prêt à être embedé.

Étapes :
1. Chargement et exploration des données brutes
2. Nettoyage du HTML dans `longdescription_fr`
3. Gestion des valeurs nulles
4. Construction du texte de corpus par événement
5. Sauvegarde dans `data/processed/events_clean.csv`

In [1]:
import pandas as pd
from bs4 import BeautifulSoup

df = pd.read_csv("../data/raw/events.csv")

print(df.shape)
df.head(3)

(5800, 9)


,uid,title_fr,longdescription_fr,conditions_fr,firstdate_begin,lastdate_end,location_name,location_address,canonicalurl
0,51542232,Diwali - Fête de la lumière,"<p>Venez célébrer la fête des lumière, Diwali ...",Tarif plein: 12€; réduit: 8€; sortir: 6€; Enfa...,2025-10-19T14:00:00+00:00,2025-10-19T20:00:00+00:00,Parc de Beauregard,"11 Avenue André Mussat, 35000 Rennes",https://openagenda.com/sortir-rennesmetropole/...
1,16590527,Séjour Changez d'Air 2026,"<p>Cette saison pour le séjour ""Changez d'air""...",Sur inscription,2026-01-06T13:00:00+00:00,2026-01-06T15:00:00+00:00,Centre socioculturel Les Longs Prés,1 rue des Longs Prés,https://openagenda.com/sortir-rennesmetropole/...
2,66886555,SAMSIC recrute pour MIX BUFFET - bus à disposi...,<p>L'agence SAMSIC vous invite à une réunion d...,NaN,2025-08-27T07:00:00+00:00,2025-08-27T10:30:00+00:00,Agence RENNES OUEST,35000 Rennes,https://openagenda.com/francetravail/events/sa...


## 1. Exploration des données brutes

On vérifie les valeurs nulles par colonne — ça détermine comment on gère les champs manquants lors de la construction du corpus.

In [2]:
df.isnull().sum()

uid                      0
title_fr               102
longdescription_fr     834
conditions_fr         2572
firstdate_begin          0
lastdate_end             0
location_name            0
location_address         0
canonicalurl             0
dtype: int64

## 2. Gestion des valeurs nulles

- `longdescription_fr` et `title_fr` : sans ces champs, l'événement ne peut pas être indexé → on supprime ces lignes
- `conditions_fr` : champ fréquemment absent (44%) mais non bloquant → on remplace par une chaîne vide

In [3]:
df = df.dropna(subset=["longdescription_fr", "title_fr"])
df["conditions_fr"] = df["conditions_fr"].fillna("")

print(f"Lignes après nettoyage des nulls : {len(df)}")

Lignes après nettoyage des nulls : 4966


## 3. Nettoyage du HTML

`longdescription_fr` contient des balises HTML (`<p>`, `<hr>`, `<a>`...). On les retire avec BeautifulSoup pour ne garder que le texte brut.

In [4]:
def strip_html(text: str) -> str:
    return BeautifulSoup(text, "html.parser").get_text(separator=" ", strip=True)

df["longdescription_fr"] = df["longdescription_fr"].apply(strip_html)

# Vérification sur un exemple
print(df["longdescription_fr"].iloc[0])

Venez célébrer la fête des lumière, Diwali avec nous, au Cadran, quartier Beauregard à Rennes. L’après midi, à partir de 16h, ateliers et animations : Hénné, Rangoli Apprendre à porter un saree Calligraphie Découverte Tabla (17h à 17h30, sur inscription, places limitées) À Partir de 18h : Cérémonie de Diwali, Concert de musique Classique d’Inde du Nord Spectacle de dance avec Mira Baï qui nous enchantera avec des danses Bollywood toutes en couleurs et chansons. Repas végétarien indien Bio sur réservation. Pour terminer la soirée, vous êtes invité à danser tous ensemble sur des chansons Bollywood, avec une initiation à la danse, animée par Mira Baï et Namasté Breizh. > Tarifs et inscriptions : L’entrée comprend les ateliers, le concert et le spectacle : 12€ / 8€ / 6€ / 3€ carte Sortir ! acceptée L’inscription est obligatoire pour la soirée, afin que nous puissions préparer la juste quantité pour le repas. Réservation des repas possible jusque vendredi 17 octobre midi. > Lieu : Le Cadran

## 4. Construction du corpus

On assemble tous les champs en un texte structuré par événement. C'est ce texte qui sera embedé puis indexé dans FAISS.

Le format structuré (avec labels) aide le modèle d'embedding à comprendre le rôle de chaque information.

In [5]:
def build_corpus(row) -> str:
    return (
        f"Titre : {row['title_fr']}\n"
        f"Description : {row['longdescription_fr']}\n"
        f"Tarif : {row['conditions_fr']}\n"
        f"Date de début : {row['firstdate_begin']}\n"
        f"Date de fin : {row['lastdate_end']}\n"
        f"Lieu : {row['location_name']}, {row['location_address']}\n"
        f"Lien : {row['canonicalurl']}"
    )

df["corpus"] = df.apply(build_corpus, axis=1)

# Vérification sur un exemple
print(df["corpus"].iloc[0])

Titre : Diwali - Fête de la lumière
Description : Venez célébrer la fête des lumière, Diwali avec nous, au Cadran, quartier Beauregard à Rennes. L’après midi, à partir de 16h, ateliers et animations : Hénné, Rangoli Apprendre à porter un saree Calligraphie Découverte Tabla (17h à 17h30, sur inscription, places limitées) À Partir de 18h : Cérémonie de Diwali, Concert de musique Classique d’Inde du Nord Spectacle de dance avec Mira Baï qui nous enchantera avec des danses Bollywood toutes en couleurs et chansons. Repas végétarien indien Bio sur réservation. Pour terminer la soirée, vous êtes invité à danser tous ensemble sur des chansons Bollywood, avec une initiation à la danse, animée par Mira Baï et Namasté Breizh. > Tarifs et inscriptions : L’entrée comprend les ateliers, le concert et le spectacle : 12€ / 8€ / 6€ / 3€ carte Sortir ! acceptée L’inscription est obligatoire pour la soirée, afin que nous puissions préparer la juste quantité pour le repas. Réservation des repas possible j

## 5. Sauvegarde

On sauvegarde le DataFrame enrichi dans `data/processed/`. Le CSV contient tous les champs originaux plus la colonne `corpus` qui sera utilisée pour l'embedding.

In [6]:
df.to_csv("../data/processed/events_clean.csv", index=False, encoding="utf-8")
print(f"Sauvegardé : {len(df)} événements dans data/processed/events_clean.csv")

Sauvegardé : 4966 événements dans data/processed/events_clean.csv
